# **Text Generation**

### Load and Inspect the CSV Dataset

In [44]:
import os
import pandas as pd
import kagglehub

if 'path' not in locals():
    path = kagglehub.dataset_download("archanghosh/robert-frost-collection")

if 'path' in locals():
    files = os.listdir(path)
    csv_files = [f for f in files if f.endswith('.csv')]
    if csv_files:
        csv_path = os.path.join(path, csv_files[0])
        poems = pd.read_csv(csv_path)
        display(poems.head(5))

,Name,Content,Collection,Year of Publication
0,NaN,NaN,NaN,NaN
1,Stopping by Woods on a Snowy Evening,Whose woods these are I think I know. \nHis ...,New Hampshire,1923.0
2,Fire and Ice,"Some say the world will end in fire,\nSome say...",New Hampshire,1923.0
3,The Aim was Song,Before man came to blow it right\nThe wind onc...,New Hampshire,1923.0
4,The Need of Being Versed in Country Things,The house had gone to bring again\nTo the midn...,New Hampshire,1923.0


### Install Necessary NLP Dependencies

In [45]:
!pip install -q transformers

### Extract and Parse Poem Lines

In [46]:
if 'poems' in locals():
    col_name = next((c for c in poems.columns if c.strip().lower() == 'content'), poems.columns[-1])
    content = poems[col_name].values
    lines = []
    for poem in content:
        if isinstance(poem, str):
            for line in poem.split("\n"):
                lines.append(line.rstrip())
    lines = [line for line in lines if len(line) > 0]
    display(lines[:5])

['Whose woods these are I think I know.',
 'His house is in the village though;',
 'He will not see me stopping here',
 'To watch his woods fill up with snow.',
 'My little horse must think it queer']

### Initialize the Text Generation Pipeline

In [47]:
from transformers import pipeline
gen = pipeline("text-generation")

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

### Print First Line Prompt

In [48]:
print(lines[0])

Whose woods these are I think I know.


### Simple Poem Completion

In [49]:
gen(lines[0], max_length=20)

[{'generated_text': 'Whose woods these are I think I know. His house is in the woods, the woods are'}]

### Multiple Returns and Length Constraints

In [50]:
gen(lines[1], max_length=30, num_return_sequences=2)

[{'generated_text': 'His house is in the village though; he rents a room there. He has a good friend, a farmer, who lives just outside the village.'},
 {'generated_text': 'His house is in the village though; he lives there with his wife and daughter.\n\nHe is a retired professor and he still likes to read. He'}]

### Clean Output Formatting using TextWrap

In [51]:
import textwrap
def wrap(x):
    return textwrap.fill(x, replace_whitespace=False, fix_sentence_endings=True)

out = gen(lines[0], max_length=30)
print(wrap(out[0]['generated_text']))

Whose woods these are I think I know.  (Robert Frost)
I think I know
the woods, I think I know the deer, the rabbit


### Prompting with Custom Queries

In [52]:
prompt = "What is an apple?"
out = gen(prompt, max_length=100)
print(wrap(out[0]['generated_text']))

What is an apple?  It's a fruit, right?  But what about a banana?
It's a fruit, too.  But what about a tomato?  Some people say it's a
fruit, some people say it's a vegetable.  So, what's the difference?
Why are some things called fruits and some things called vegetables?
The answer lies in the definition of a fruit and a vegetable.

A fruit
is the mature ovary of a flowering plant.  That's right, a fruit is
just the
